In [5]:

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

# Get the absolute path to the directory containing the current script
try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from types import SimpleNamespace as ns
from dcc_scripts.vis.condition_plot_specs import resolve_plot_set
from src.analysis.utils.general_utils import (
    get_sig_chans_per_subject,
    make_or_load_subjects_electrodes_to_ROIs_dict,
    make_sig_electrodes_per_subject_and_roi_dict,
)

args = ns(
    # these are actually exported env vars from the submit script
    plot_sets = 'all_lpfc, task_relevant_lpfc',
    plot_set_label = 'lpfc_power_trace_sets',
    power_figs_base = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs",
    anova_epochs_root = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_and_nan_thresh_perc_5.0_70.0-150.0_Hz_padLength_1.5s_filterbank_hilbert_stat_func_ttest_zmax_20",
    anova_unit = 'electrode',
    anova_labels_csv = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/stats/results/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_1.5s_filterbank_hilbert_stat_func_ttest_ind_equal_var_False_nan_policy_omit/anova_conjunction_window_0.0to1.5s_sig_lpfc_condition_none/anova_labels.csv",

    # these are run script args
    LAB_root = None,
    # subjects = ['D0057', 'D0059', 'D0063', 'D0065', 'D0069', 'D0077', 'D0090', 'D0094', 'D0100', 'D0102', 'D0103', 'D0107A', 'D0110', 'D0116', 'D0117', 'D0121', 'D0133', 'D0134', 'D0137', 'D0138', 'D0139A', 'D0144', 'D0145', 'D0146'],
    subjects = ['D0130'],
    
    electrodes = 'sig',
    task = 'GlobalLocal',
    
    save_dir = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/vis/figs/sandbox",
    config_dir = os.path.join(project_root, "src", "analysis", "config"),
    hemi = 'both',
    zoom = None,
    marker_size = 0.1,
    transparency = 0.4,
    rm_wm = False,
    
    make_histograms = True,
    hist_drop_white_matter = True,
    hist_top_n = None,
    
    rois_dict = {
        'lpfc': ["G_front_inf-Opercular", "G_front_inf-Orbital", "G_front_inf-Triangul", "G_front_middle", "G_front_sup", "Lat_Fis-ant-Horizont", "Lat_Fis-ant-Vertical", "S_circular_insula_ant", "S_circular_insula_sup", "S_front_inf", "S_front_middle", "S_front_sup"]
    },
    acc_trials_only = True,
    anova_label_effect = 'lwpc',
    anova_label_correction = 'flags',
    n_jobs=-1
)

plot_set = resolve_plot_set(args.plot_set_label)


In [ ]:

#!/usr/bin/env python
"""
Plot significant electrodes for one or more conditions on the fsaverage brain,
each condition in its own color, and make histograms of the anatomical ROIs
that those electrodes fall in.

This is the cleaned-up, submittable version of ``src/analysis/vis/plot_subjects.ipynb``.
It is driven by an ``args`` namespace (see ``run_plot_sig_electrodes_dcc.py``) so it
can be launched on the cluster via sbatch, or imported and called interactively.

For each condition the caller supplies:
  - an ``epochs_root_file`` that identifies the ``sig_chans_{subject}_{epochs_root_file}.json``
    files written by the stats pipeline, and
  - a ``color`` (matplotlib RGB tuple) to draw that condition's electrodes in.

Outputs (written to ``args.save_dir``):
  - ``brain_<combined_name>.png``            : all conditions overlaid on one brain.
  - ``brain_<condition>.png``                : one brain per condition.
  - ``roi_hist_<condition>.png``             : ROI histogram for each condition.
  - ``sig_electrodes_<condition>.json``      : the electrodes plotted, for reproducibility.

The heavy lifting (loading sig chans, mapping electrodes to ROIs) reuses the
shared helpers in ``src.analysis.utils.general_utils`` so this stays in sync with
the rest of the pipeline.
"""

import os
import re
import json
from collections import Counter, OrderedDict

import matplotlib
matplotlib.use("Agg")  # headless: never open a window on the cluster
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Headless 3D rendering. Compute nodes have no physical display, so there are
# two ways to render the brain, and they have to be decided before the brain
# backend is imported:
#
#   1. ``xvfb-run`` gave us a virtual DISPLAY (what the sbatch script does).
#      Let MNE build its Qt window on it as usual: the window is realized, so
#      its OpenGL context is current and screenshots work everywhere.
#   2. No DISPLAY at all. Fall back to PyVista's off-screen mode; the window is
#      never realized, so ``save_brain_image`` below has to make the render
#      window current by hand before screenshotting it.
# ---------------------------------------------------------------------------
HAVE_DISPLAY = True

# Use the project's copy of these helpers, not ieeg's: ieeg's version resolves
# the recon directory to ``~/Box/ECoG_Recon``, which doesn't exist on the
# cluster. ``jim_mri.get_sub_dir`` points at the DCC recon directory instead, so
# ``subject_to_info`` and ``plot_on_average`` must come from the same module or
# they'd read electrode locations from two different places.
from src.analysis.vis.jim_mri import plot_on_average, subject_to_info
from src.analysis.utils.general_utils import (
    get_sig_chans_per_subject,
    make_or_load_subjects_electrodes_to_ROIs_dict,
    make_sig_electrodes_per_subject_and_roi_dict,
)
from src.analysis.power.power_traces import load_significant_electrodes
from src.analysis.utils.anova_label_selection import load_anova_label_electrodes

# Regex that strips leading zeros from a subject id: 'D0057' -> 'D57', 'D0107A' -> 'D107A'.
_SUBJECT_PATTERN = re.compile(r"^D(0*)(\d+)([A-Za-z]*)$")


# ===========================================================================
# Helpers
# ===========================================================================
def strip_leading_zeros(subject_id):
    """'D0057' -> 'D57', 'D0107A' -> 'D107A'. Unrecognized ids are returned as-is."""
    match = _SUBJECT_PATTERN.match(subject_id)
    if not match:
        return subject_id
    _, numbers, letters = match.groups()
    return f"D{int(numbers)}{letters}"


def save_brain_image(fig, path):
    """Screenshot a :class:`mne.viz.Brain` to ``path``, headless-safe.

    ``Brain.save_image`` grabs the plotter's framebuffer. On a headless node the
    window is never realized, and pyvista < 0.48 doesn't make the render window
    current before reading it, so the grab dies with
    ``RenderWindowUnavailable: Render window is not current``. The first
    ``save_image`` call still performs the initial render, so on that failure we
    make the window current ourselves -- what newer pyvista does internally --
    and grab again.
    """
    try:
        fig.save_image(path)
        return True
    except Exception as first_error:
        render_window = getattr(getattr(fig, "plotter", None), "render_window", None)
        if render_window is None:
            print(f"  Could not save {path}: {first_error}")
            return False
        print(f"  Screenshot failed ({first_error}); making the render window "
              f"current and retrying.")
        try:
            render_window.MakeCurrent()
            fig.save_image(path)
            return True
        except Exception as retry_error:
            print(f"  Could not save {path}: {retry_error}")
            return False


def collapse_rois_to_subject_dict(sig_electrodes_per_subject_roi):
    """Collapse a {roi: {subject: [electrodes]}} dict into {subject: [electrodes]}.

    Duplicate electrodes (an electrode that lands in more than one ROI) are kept
    only once.
    """
    collapsed = {}
    for _roi, subject_dict in sig_electrodes_per_subject_roi.items():
        for subject, electrodes in subject_dict.items():
            bucket = collapsed.setdefault(subject, [])
            bucket.extend(e for e in electrodes if e not in bucket)
    return collapsed


def tuples_to_subject_dict(sub_elec_tuples):
    """Group ``[(subject, electrode), ...]`` into ``{subject: [electrodes]}``."""
    grouped = {}
    for subject, electrode in sub_elec_tuples:
        bucket = grouped.setdefault(subject, [])
        if electrode not in bucket:
            bucket.append(electrode)
    return grouped


def get_condition_electrodes(subjects, cfg, task, LAB_root,
                             rois_dict=None, subjects_electrodes_to_ROIs_dict=None,
                             config_dir=None):
    """Significant electrodes for one condition, as {subject_with_zeros: [names]}.

    A condition is described by a config dict ``cfg`` and may draw its electrodes
    from one of two sources:

    1. **Per-condition sig_chans files** (``cfg['epochs_root_file']``):
       reads ``sig_chans_{subject}_{epochs_root_file}.json``. Optionally
       restricted to ``rois_dict``. Good for single-contrast effects (e.g. a
       congruency or response contrast).

    2. **A within-electrode ANOVA run, filtered by effect**
       (``cfg['anova_run_dir']`` + ``cfg['effect']``): reads the run's
       ``significant_effects_structure.json`` / ``summary.csv`` via
       :func:`load_significant_electrodes`. This is how interaction effects such
       as LWPC (``C(congruency):C(incongruentProportion)``) and LWPS
       (``C(switchType):C(switchProportion)``) are identified. Optional keys:
       ``use_fdr`` (default True), ``p_thresh`` (default 0.05), ``anova_roi``
       (restrict to one ANOVA ROI, default all).

    Parameters
    ----------
    subjects : list of str
        Subject ids with leading zeros, e.g. ``['D0057', 'D0059']``.
    cfg : dict
        Condition config. Must contain either ``'epochs_root_file'`` or
        ``'anova_run_dir'`` (with ``'effect'``), plus a ``'color'`` used later.
    task, LAB_root :
        Passed through to :func:`get_sig_chans_per_subject`.
    rois_dict : dict, optional
        For the sig_chans source only: restrict to these anatomical ROIs.
        Ignored for the ANOVA source, which is already ROI-scoped by the run.
    subjects_electrodes_to_ROIs_dict : dict, optional
        Prebuilt electrode->ROI mapping (loaded from ``config_dir`` if needed).
    config_dir : str, optional
        Where the ``subjects_electrodestoROIs_dict.json`` lives.

    Returns
    -------
    dict
        ``{subject_with_zeros: [electrode_names]}``.
    """
    # ---- Source 0a: saved A1/window-mean ANOVA labels -----------------------
    if "anova_labels_csv" in cfg:
        if not cfg["anova_labels_csv"]:
            raise ValueError(
                "This *_labels plot set needs ANOVA_LABELS_CSV to point to the "
                "source anova_labels.csv (not an anova_label_selections figures directory).")
        selected = load_anova_label_electrodes(
            cfg["anova_labels_csv"], effect=cfg["anova_label_effect"],
            correction=cfg.get("anova_label_correction", "flags"),
            alpha=cfg.get("anova_label_alpha", 0.05),
            roi=cfg.get("anova_label_roi"))
        return collapse_rois_to_subject_dict(selected)

    # ---- Source 0: every electrode in the requested anatomical ROI ----------
    if cfg.get("all_roi"):   
        if subjects_electrodes_to_ROIs_dict is None:
            subjects_electrodes_to_ROIs_dict = make_or_load_subjects_electrodes_to_ROIs_dict(
                subjects, task=task, LAB_root=LAB_root, save_dir=config_dir,
                filename="subjects_electrodestoROIs_dict.json")
            
        if not rois_dict: # new change 10/28/26
            all_roi = subjects_electrodes_to_ROIs_dict
            return collapse_rois_to_subject_dict(all_roi)
        
        all_roi, _ = make_sig_electrodes_per_subject_and_roi_dict(
            rois_dict, subjects_electrodes_to_ROIs_dict,
            {subject: [] for subject in subjects})
        return collapse_rois_to_subject_dict(all_roi)

    # ---- Source 2a: set algebra over power-trace ANOVA effects --------------
    if cfg.get("anova_run_dir") and (
            cfg.get("include_effects") or cfg.get("exclude_effects")):
        run_dir = cfg["anova_run_dir"]
        def effect_set(effect):
            return set(load_significant_electrodes(
                run_dir, roi=cfg.get("anova_roi"), effect=effect,
                use_fdr=cfg.get("use_fdr", True),
                p_thresh=cfg.get("p_thresh", 0.05)))

        included = [effect_set(effect)
                    for effect in cfg.get("include_effects", ())]
        selected = set.intersection(*included) if included else set()
        for effect in cfg.get("exclude_effects", ()):
            selected -= effect_set(effect)
        return tuples_to_subject_dict(sorted(selected))

    # ---- Source 2b: within-electrode ANOVA run, filtered by effect ----------
    if cfg.get("anova_run_dir"):
        sub_elec = load_significant_electrodes(
            cfg["anova_run_dir"],
            roi=cfg.get("anova_roi"),
            effect=cfg.get("effect"),
            use_fdr=cfg.get("use_fdr", True),
            p_thresh=cfg.get("p_thresh", 0.05),
        )
        return tuples_to_subject_dict(sub_elec)

    # ---- Source 1: per-condition sig_chans files ---------------------------
    sig_chans_per_subject = get_sig_chans_per_subject(
        subjects, cfg["epochs_root_file"], task=task, LAB_root=LAB_root)

    if not rois_dict:
        # Whole brain: keep every significant channel.
        return {subj: list(chans) for subj, chans in sig_chans_per_subject.items()}

    if subjects_electrodes_to_ROIs_dict is None:
        subjects_electrodes_to_ROIs_dict = make_or_load_subjects_electrodes_to_ROIs_dict(
            subjects, task=task, LAB_root=LAB_root, save_dir=config_dir,
            filename="subjects_electrodestoROIs_dict.json")

    _all, sig_electrodes_per_subject_roi = make_sig_electrodes_per_subject_and_roi_dict(
        rois_dict, subjects_electrodes_to_ROIs_dict, sig_chans_per_subject)
    return collapse_rois_to_subject_dict(sig_electrodes_per_subject_roi)


def build_global_index_map(subjects_no_zeros, subjects_dir=None):
    """Map each subject to its offset in the concatenated fsaverage channel space.

    ``plot_on_average`` addresses electrodes by a single flat index that runs
    across all subjects (in the order they are passed). Using one shared map for
    every condition keeps the indices comparable, which is what lets us detect
    electrodes shared by multiple conditions.

    Parameters
    ----------
    subjects_no_zeros : list of str
        Subject recon codes, e.g. ``['D57', 'D59']``.
    subjects_dir : PathLike, optional
        Recon directory holding ``<subject>/elec_recon/``. Defaults to
        :func:`jim_mri.get_sub_dir` (the DCC recon directory).

    Returns
    -------
    offsets : OrderedDict
        ``{subject_no_zeros: (offset, ch_names)}``.
    """
    offsets = OrderedDict()
    running = 0
    for subject in subjects_no_zeros:
        info = subject_to_info(subject, subjects_dir=subjects_dir)
        offsets[subject] = (running, info["ch_names"])
        running += len(info["ch_names"])
    return offsets


def electrodes_to_global_indices(condition_electrodes, offsets):
    """Convert {subject_with_zeros: [names]} into a sorted list of global indices.

    Electrodes that are not present in a subject's fsaverage info (e.g. dropped
    during preprocessing) are skipped with a warning.
    """
    indices = []
    for subject_with_zeros, electrodes in condition_electrodes.items():
        subject = strip_leading_zeros(subject_with_zeros)
        if subject not in offsets:
            print(f"  Warning: {subject_with_zeros} not in the plotting subject list; skipping.")
            continue
        offset, ch_names = offsets[subject]
        for electrode in electrodes:
            if electrode in ch_names:
                indices.append(offset + ch_names.index(electrode))
            else:
                print(f"  Warning: electrode '{electrode}' not found in {subject}; skipping.")
    return sorted(set(indices))


def split_unique_and_overlap(condition_indices):
    """Split condition index sets into per-condition-unique and shared-overlap.

    Parameters
    ----------
    condition_indices : OrderedDict
        ``{condition_name: set_of_global_indices}``.

    Returns
    -------
    unique : OrderedDict
        ``{condition_name: [indices in this condition and no other]}``.
    overlap : list
        Indices that appear in two or more conditions.
    """
    counts = Counter()
    for idx_set in condition_indices.values():
        counts.update(idx_set)
    overlap = sorted(idx for idx, n in counts.items() if n > 1)
    overlap_set = set(overlap)
    unique = OrderedDict(
        (name, sorted(i for i in idx_set if i not in overlap_set))
        for name, idx_set in condition_indices.items()
    )
    return unique, overlap


# ===========================================================================
# ROI histogram
# ===========================================================================
def electrode_roi_counts(condition_electrodes, subjects_electrodes_to_ROIs_dict,
                         drop_white_matter=True):
    """Count the anatomical ROI of every electrode in a condition group.

    Uses the per-subject ``default_dict`` (electrode -> anatomical label).
    """
    counts = Counter()
    for subject_with_zeros, electrodes in condition_electrodes.items():
        subj_entry = subjects_electrodes_to_ROIs_dict.get(subject_with_zeros, {})
        default_dict = subj_entry.get("default_dict", {})
        for electrode in electrodes:
            roi = default_dict.get(electrode, "Unknown")
            if drop_white_matter and ("White-Matter" in roi or roi in ("Unknown", "unknown")):
                continue
            counts[roi] += 1
    return counts


def plot_roi_histogram(counts, title, save_path, color=(0.2, 0.4, 0.8), top_n=None):
    """Horizontal bar chart of ROI counts, sorted most-common first."""
    if not counts:
        print(f"  No ROIs to plot for '{title}'; skipping histogram.")
        return
    items = counts.most_common(top_n)
    labels = [label for label, _ in items][::-1]   # reverse so largest is on top
    values = [value for _, value in items][::-1]

    height = max(3.0, 0.35 * len(labels) + 1.5)
    fig, ax = plt.subplots(figsize=(10, height))
    ax.barh(labels, values, color=color, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Number of significant electrodes")
    ax.set_title(title)
    for y, value in enumerate(values):
        ax.text(value, y, f" {value}", va="center", fontsize=9)
    ax.margins(x=0.08)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved ROI histogram -> {save_path}")


Qt5Agg backend not available, using default backend


In [2]:
# ===========================================================================
# Main
# ===========================================================================
def main(args):
    """Run the plotting pipeline.

    Expected attributes on ``args`` (see ``run_plot_sig_electrodes_dcc.py``):
        subjects                : list[str]        subject ids with leading zeros
        conditions              : OrderedDict      {name: cfg}. Each cfg has a
                                  'color' (r,g,b) plus ONE electrode source:
                                    - {'epochs_root_file': str}                 sig_chans files
                                    - {'anova_run_dir': str, 'effect': str, ...} within-elec ANOVA
        task                    : str
        LAB_root                : str | None
        rois_dict               : dict | None      restrict to these ROIs, or None for whole brain
        config_dir              : str              where subjects_electrodestoROIs_dict.json lives
        subjects_dir            : str | None       recon dir (<subject>/elec_recon/); None = jim_mri default
        save_dir                : str              output directory (created if missing)
        hemi                    : str              'both' | 'lh' | 'rh' | 'split'
        zoom                    : float | None     per-panel camera zoom (<1 zooms out);
                                                   None = renderer default, which keeps
                                                   the two 'split' panels apart
        size                    : float            electrode marker size
        transparency            : float            brain transparency
        rm_wm                   : bool             drop white-matter electrodes when plotting
        mutually_exclusive      : bool             plot shared electrodes in overlap_color
        overlap_color           : (r,g,b)          color for electrodes shared by >1 condition
        make_histograms         : bool
        combined_name           : str              filename stem for the combined brain figure
    """
    os.makedirs(args.save_dir, exist_ok=True)
    print(f"Saving figures to: {args.save_dir}")

    subjects = list(args.subjects)
    subjects_no_zeros = [strip_leading_zeros(s) for s in subjects]

    rois_dict = getattr(args, "rois_dict", None)
    make_histograms = getattr(args, "make_histograms", True)
    subjects_dir = getattr(args, "subjects_dir", None)

    # Electrode -> ROI mapping is needed for ROI filtering and for histograms.
    subjects_electrodes_to_ROIs_dict = None
    if rois_dict or make_histograms:
        subjects_electrodes_to_ROIs_dict = make_or_load_subjects_electrodes_to_ROIs_dict(
            subjects, task=args.task, LAB_root=args.LAB_root, save_dir=args.config_dir,
            filename="subjects_electrodestoROIs_dict.json")

    # ---- 1. Load each condition's significant electrodes -------------------
    condition_electrodes = OrderedDict()   # {name: {subj_with_zeros: [names]}}
    for name, cfg in args.conditions.items():
        print(f"\n=== Loading significant electrodes for condition: {name} ===")
        electrodes = get_condition_electrodes(
            subjects, cfg, task=args.task, LAB_root=args.LAB_root,
            rois_dict=rois_dict,
            subjects_electrodes_to_ROIs_dict=subjects_electrodes_to_ROIs_dict,
            config_dir=args.config_dir)
        condition_electrodes[name] = electrodes
        n = sum(len(v) for v in electrodes.values())
        print(f"  {n} electrodes across {len(electrodes)} subjects.")

        # Save the plotted electrodes for reproducibility.
        out_json = os.path.join(args.save_dir, f"sig_electrodes_{name}.json")
        with open(out_json, "w") as fh:
            json.dump(electrodes, fh, indent=2)

    # ---- 2. Convert to a shared global index space -------------------------
    offsets = build_global_index_map(subjects_no_zeros, subjects_dir=subjects_dir)
    condition_indices = OrderedDict(
        (name, set(electrodes_to_global_indices(elecs, offsets)))
        for name, elecs in condition_electrodes.items()
    )

    mutually_exclusive = getattr(args, "mutually_exclusive", True)
    overlap_color = getattr(args, "overlap_color", (0, 0, 0))

    if mutually_exclusive:
        plot_groups, overlap = split_unique_and_overlap(condition_indices)
    else:
        plot_groups, overlap = condition_indices, []

    # Showing the window is what realizes its OpenGL context, so do it whenever
    # there's a display (xvfb's counts); without one we're in off-screen mode
    # and there is nothing to show.
    show_brain = HAVE_DISPLAY
    failed_figures = []
    brain_zoom = getattr(args, "zoom", None)

    # ---- 3. Combined brain: every condition overlaid on one figure ---------
    print("\n=== Plotting combined brain figure ===")
    fig = None
    for name, indices in plot_groups.items():
        if not indices:
            print(f"  {name}: no electrodes to plot.")
            continue
        color = args.conditions[name]["color"]
        fig = plot_on_average(
            subjects_no_zeros, subj_dir=subjects_dir, picks=sorted(indices),
            rm_wm=args.rm_wm, hemi=args.hemi, color=color, size=args.size,
            transparency=args.transparency, fig=fig, show=show_brain,
            zoom=brain_zoom)
    if mutually_exclusive and overlap:
        print(f"  overlap: {len(overlap)} electrodes shared by >1 condition.")
        fig = plot_on_average(
            subjects_no_zeros, subj_dir=subjects_dir, picks=overlap,
            rm_wm=args.rm_wm, hemi=args.hemi, color=overlap_color,
            size=args.size, transparency=args.transparency, fig=fig,
            show=show_brain, zoom=brain_zoom)

    if fig is not None:
        combined_name = getattr(args, "combined_name", "combined")
        combined_path = os.path.join(args.save_dir, f"brain_{combined_name}.png")
        if save_brain_image(fig, combined_path):
            print(f"  Saved combined brain -> {combined_path}")
        else:
            failed_figures.append(combined_path)
        fig.close()

    # ---- 4. One brain per condition (in that condition's color) ------------
    for name, indices in condition_indices.items():
        if not indices:
            continue
        color = args.conditions[name]["color"]
        cfig = plot_on_average(
            subjects_no_zeros, subj_dir=subjects_dir, picks=sorted(indices),
            rm_wm=args.rm_wm, hemi=args.hemi, color=color, size=args.size,
            transparency=args.transparency, show=show_brain, zoom=brain_zoom)
        cpath = os.path.join(args.save_dir, f"brain_{name}.png")
        if save_brain_image(cfig, cpath):
            print(f"  Saved brain for {name} -> {cpath}")
        else:
            failed_figures.append(cpath)
        cfig.close()

    # ---- 5. ROI histograms -------------------------------------------------
    if make_histograms:
        print("\n=== Plotting ROI histograms ===")
        for name, electrodes in condition_electrodes.items():
            counts = electrode_roi_counts(
                electrodes, subjects_electrodes_to_ROIs_dict,
                drop_white_matter=getattr(args, "hist_drop_white_matter", True))
            color = args.conditions[name]["color"]
            hist_path = os.path.join(args.save_dir, f"roi_hist_{name}.png")
            plot_roi_histogram(
                counts, title=f"ROIs of significant electrodes: {name}",
                save_path=hist_path, color=color,
                top_n=getattr(args, "hist_top_n", None))

    # The histograms above don't depend on the 3D renderer, so they're worth
    # producing even if the brain screenshots failed -- but don't call the run a
    # success in that case.
    if failed_figures:
        raise RuntimeError(
            "Could not screenshot the brain figure(s): "
            + ", ".join(failed_figures)
            + ". The electrodes were resolved fine, so this is a rendering "
              "problem: run under 'xvfb-run -a' (see sbatch_plot_sig_electrodes_dcc.sh).")

    print("\nDone.")

In [13]:
subjects_electrodes_to_ROIs_dict

{'D0057': {'default_dict': {'RAI1': 'ctx_rh_G_Ins_lg_and_S_cent_ins',
   'RAI2': 'Right-Cerebral-White-Matter',
   'RAI3': 'Right-Cerebral-White-Matter',
   'RAI4': 'Right-Cerebral-White-Matter',
   'RAI5': 'Right-Cerebral-White-Matter',
   'RAI6': 'ctx_rh_S_circular_insula_sup',
   'RAI7': 'Right-Cerebral-White-Matter',
   'RAI8': 'Right-Cerebral-White-Matter',
   'RAI9': 'Right-Cerebral-White-Matter',
   'RAI10': 'Right-Cerebral-White-Matter',
   'RAI11': 'Right-Cerebral-White-Matter',
   'RAI12': 'ctx_rh_S_front_sup',
   'RAI13': 'ctx_rh_S_front_sup',
   'RAI14': 'ctx_rh_S_front_sup',
   'RAI15': 'ctx_rh_G_front_sup',
   'RAI16': 'ctx_rh_G_front_sup',
   'RPI1': 'Right-Cerebral-White-Matter',
   'RPI2': 'Right-Cerebral-White-Matter',
   'RPI3': 'Right-Cerebral-White-Matter',
   'RPI4': 'Right-Cerebral-White-Matter',
   'RPI5': 'Right-Cerebral-White-Matter',
   'RPI6': 'Right-Cerebral-White-Matter',
   'RPI7': 'Right-Cerebral-White-Matter',
   'RPI8': 'Right-Cerebral-White-Matter',
 

In [12]:
subjects_electrodes_to_ROIs_dict = make_or_load_subjects_electrodes_to_ROIs_dict(args.subjects, task=args.task, LAB_root=args.LAB_root, 
                                                                                 save_dir=args.config_dir,
                                                                                 filename="subjects_electrodestoROIs_dict.json")



all_roi, _ = make_sig_electrodes_per_subject_and_roi_dict(
    args.rois_dict, subjects_electrodes_to_ROIs_dict,
    {subject: [] for subject in args.subjects})
x = collapse_rois_to_subject_dict(all_roi)

x

Attempting to load the subjects' electrodes-to-ROIs dictionary...
Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Dictionary loaded successfully. Ready to proceed!
For subject D0057, G_front_inf-Opercular, G_front_inf-Orbital, G_front_inf-Triangul, G_front_middle, G_front_sup, Lat_Fis-ant-Horizont, Lat_Fis-ant-Vertical, S_circular_insula_ant, S_circular_insula_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RAI6', 'RAI12', 'RAI13', 'RAI14', 'RAI15', 'RAI16', 'RPI15', 'RPI14', 'RAMF10', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14', 'RAIF11', 'RAIF12', 'RAIF13', 'RAIF14']
For subject D0059, G_front_inf-Opercular, G_front_inf-Orbital, G_front_inf-Triangul, G_front_middle, G_front_sup, Lat_Fis-ant-Horizont, Lat_Fis-ant-Vertical, S_circular_insula_ant, S_circular_insula_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LMMF9', 'LMMF11', 'LMMF10', 'LMMF12', 'LPSF16']
For subject D0063, G_front_inf-Opercula

{'D0057': ['RAI6',
  'RAI12',
  'RAI13',
  'RAI14',
  'RAI15',
  'RAI16',
  'RPI15',
  'RPI14',
  'RAMF10',
  'RAMF11',
  'RAMF12',
  'RAMF13',
  'RAMF14',
  'RAIF11',
  'RAIF12',
  'RAIF13',
  'RAIF14'],
 'D0059': ['LMMF9', 'LMMF11', 'LMMF10', 'LMMF12', 'LPSF16'],
 'D0063': ['LASF10',
  'LASF11',
  'LASF14',
  'LASF15',
  'LASF16',
  'LMSF5',
  'LMSF6',
  'LMSF12',
  'LPSF10',
  'LPSF12',
  'ROF16',
  'RAI16',
  'RAMF11',
  'RAMF12',
  'RAMF13',
  'RAMF14',
  'RMMF13',
  'RMMF14',
  'RAI4',
  'RAI6',
  'RAI5',
  'RAI10',
  'RAI11',
  'RASF15',
  'RASF16',
  'RMSF8',
  'RMSF9',
  'RMSF10',
  'RMSF11',
  'RMSF12',
  'RMSF7',
  'RAMF8',
  'RAMF9',
  'RAMF10',
  'RMMF9',
  'RMMF10'],
 'D0065': ['RASF13',
  'RASF14',
  'RASF15',
  'RASF16',
  'RMSF11',
  'RMSF12',
  'RMSF13',
  'RMSF14',
  'RI7'],
 'D0069': ['LOF8'],
 'D0071': ['RFO14',
  'RFO16',
  'RIA4',
  'RIP6',
  'RIA5',
  'RIA11',
  'RIA12',
  'RIA13',
  'RIA14',
  'RIA16',
  'RIP14',
  'RIP15',
  'RIP16'],
 'D0077': [],
 'D0090': [

In [ ]:
subjects_electrodes_to_ROIs_dict

In [ ]:
# # ---- Source 0: every electrode in the requested anatomical ROI ----------
# if cfg.get("all_roi"):
#     if not rois_dict:
        
#         raise ValueError("all_roi requires a rois_dict")
#         # TODO: this needs to be swapped to return all electrodes from the specified subjects list 
#         # #instead of raising a ValueError. Grab elecs from default_dict. Check how make_or_load_subjects_electrodes_to_ROIs_dict works.
#         # and go ahead and return this rather than the sig electrodes dict.
#         # actually, for lpfc too, i want to return all elecs rather than just the sig elecs. so also need to return all elecs for that.
        
#     if subjects_electrodes_to_ROIs_dict is None:
#         subjects_electrodes_to_ROIs_dict = make_or_load_subjects_electrodes_to_ROIs_dict(
#             subjects, task=task, LAB_root=LAB_root, save_dir=config_dir,
#             filename="subjects_electrodestoROIs_dict.json")
#     all_roi, _ = make_sig_electrodes_per_subject_and_roi_dict(
#         rois_dict, subjects_electrodes_to_ROIs_dict,
#         {subject: [] for subject in subjects})
#     return collapse_rois_to_subject_dict(all_roi)

In [ ]:
x = make_or_load_subjects_electrodes_to_ROIs_dict

In [7]:
import pandas as pd
json_dict = pd.read_json('/hpc/home/jz421/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json')
test = list(json_dict.keys())